**[🏠 Course Home](../README.md)** | **📖 [Conceptual Track](../conceptual/17_tolerance_intervals_and_decision_sizing.ipynb)** | **🤖 [AI Track](17_tolerance_intervals_and_decision_sizing.ipynb)** | **🐍 [Python Track](../python/17_tolerance_intervals_and_decision_sizing.ipynb)** | **🔵 [R Track](../r/17_tolerance_intervals_and_decision_sizing.ipynb)** | **⬅️ [Prev: Ch 16: Decision Theory](16_bayesian_decision_theory_and_predictive_checks.ipynb)** | **➡️ [Next: Ch 18: Flaky Tests & Pipelines](18_case_studies_flaky_tests_and_pipelines.ipynb)**

*🧭 Part IV: Dynamic Systems & Decision Theory — Chapter 17 of 22*

---

# Chapter 17: Tolerance Intervals & Decision Sizing
### *Prompting for Population Guarantees, The Plug-In Fallacy, and ISO Sizing*

---

> **🧭 Course Context & Why This Module Matters in the AI Era:**
>
> When infrastructure architects, Site Reliability Engineers, and hardware
> designers size systems, they must provide contractual guarantees:
> * *"What timeout threshold $T$ guarantees that at least $99\%$ of all future
>   checkout requests finish in time, with $95\%$ statistical confidence?"*
> * *"How much memory buffer must we allocate to prevent Out-Of-Memory
>   kills on $99.9\%$ of workloads?"*
>
> When engineers prompt Large Language Models (LLMs) to compute these capacity
> limits from sample telemetry (e.g., $n = 15$ test runs), AI assistants
> almost universally fall into **The Plug-In Fallacy**:
> $$T_{\text{naive}} = \bar{x} + z_{0.99} \cdot s = 150 + 2.326 \times 20 = \mathbf{196.5\text{ ms}}$$
>
> The Plug-In Fallacy pretends that sample statistics $(\bar{x}, s)$ are the
> immutable true parameters $(\mu, \sigma)$. For small samples ($n = 15$),
> this ignores parameter estimation uncertainty, producing capacity limits that
> fail to cover the required population percentile in over **$40\%$ of
> real-world deployments**.
>
> This module teaches you how to prompt AI agents to compute ISO 16269-6
> tolerance factors and Bayesian posterior predictive tolerance bounds.


## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

### The Prompt That Fools the AI

A cloud infrastructure lead needs to configure an API gateway timeout for a
new database microservice. Telemetry from $n = 15$ preliminary benchmark runs
yields:
$$\text{Sample Mean Latency } \bar{x} = 150.0\text{ ms}, \qquad \text{Sample Standard Deviation } s = 20.0\text{ ms}$$

The executive SLA requirement states:
> *"Guarantee with 95% confidence that at least 99% of all future production
> requests complete in under $T$ milliseconds."*

The engineer submits this prompt to an AI assistant:

```text
We measured database latency across 15 benchmark runs: mean = 150ms, std = 20ms.
Write a Python script to compute the timeout limit T that guarantees with 95%
confidence that at least 99% of all future requests finish in time.
```

---

### The Plausible AI Hallucination

The AI produces this standard Python snippet:

```python
# AI Generated Timeout Calculation (The Plug-In Trap!)
from scipy import stats

x_bar = 150.0
s = 20.0
p = 0.99

# Standard normal quantile
z_score = stats.norm.ppf(p)  # 2.3263
timeout_T = x_bar + z_score * s

print(f"Calculated 99% Latency Guarantee: {timeout_T:.2f} ms")
```

The AI confidently reports:
> *"Under the normal distribution, 99% of requests fall within 2.326 standard
> deviations of the mean. Therefore, setting your timeout to **196.53 ms**
> guarantees that 99% of future requests will complete in time with 95%
> confidence!"*

---

### Forensic Post-Mortem: Why the System Crashes

```
           THE PLUG-IN DISASTER VS. ISO TOLERANCE REALITY
 Latency
  (ms)
   230 |
   220 |---------------------------------------● [ISO 16269-6 Bound: 220.4ms]
   210 |                                         (Real 95% Conf / 99% Coverage)
   205 |                           * [Prediction Interval: 204.8ms]
   196 |=======================================  [AI Plug-In Claim: 196.5ms]
       |                                         (Covers only ~96.5% of fleet!)
   170 |
   150 |-------------------●-------------------  [Sample Mean: 150.0ms]
       +---------------------------------------
                          n = 15
```

#### 1. What 99% of Engineers and AIs Confuse: The Three Intervals
* **Confidence Interval**: Where is the population center $\mu$?
  $$\bar{x} \pm t_{n-1} \cdot \frac{s}{\sqrt{n}} \quad (\text{Width} \to 0 \text{ as } n \to \infty)$$
* **Prediction Interval**: Where is the *single next request* $y_{n+1}$?
  $$\bar{x} \pm t_{n-1} \cdot s \sqrt{1 + \frac{1}{n}}$$
* **Tolerance Interval**: Where does a fraction $P = 99\%$ of the
  *entire future population* live, with confidence $\gamma = 95\%$?

#### 2. The Plug-In Catastrophe
The AI assumed $\mu = 150$ and $\sigma = 20$. But with only $n = 15$:
* The standard error of the mean is $s / \sqrt{15} = 5.16\text{ ms}$.
* The sample standard deviation $s$ has wide sampling dispersion:
  $$\frac{(n-1)s^2}{\sigma^2} \sim \chi^2_{14}$$
  There is a $30\%$ chance that $s$ underestimated the true standard deviation
  $\sigma$ by more than $15\%$.

#### 3. The ISO 16269-6 Exact Solution
Under rigorous non-central $t$-distribution tolerance statistics:
$$k_2(n=15,\, P=0.99,\, \gamma=0.95) \approx \mathbf{3.520}$$
The true required timeout threshold is:
$$T_{\text{ISO}} = \bar{x} + k_2 \cdot s = 150.0 + 3.520 \times 20.0 = \mathbf{220.40\text{ ms}}$$

By provisioning at $196.53\text{ms}$, the infrastructure under-provisioned by
nearly **$24\text{ ms}$**! In production simulations across 10,000 fleets, the
AI's naive threshold failed to achieve $99\%$ coverage in over $42\%$ of fleets,
triggering severe SLA contract breaches.


## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. Data: Small benchmark sample of n = 15 service latencies (x_bar = 150 ms,
   s = 20 ms).
2. Guarantee: Provide a Two-Tier (P = 0.99, 1 - alpha = 0.95) tolerance guarantee
   covering 99% of future requests with 95% confidence.
3. Invariant: Use ISO 16269-6 non-central t factor (k ~ 3.52) or Bayesian posterior
   predictive simulation; reject naive plug-in x_bar + 2.326 s.
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
ISO Factor: UTL = x_bar + k(n, P, 1-alpha) * s
k-factor:   k = qt(1-alpha, df=n-1, ncp=z_P * sqrt(n)) / sqrt(n)
Bayes:      95th percentile of 99th quantile of y_new ~ Normal(mu^(s), sigma^(s))
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as an Infrastructure Capacity Statistician. I have 15 benchmark runs and
need to set a latency timeout that protects 99% of users. Ask me 3 questions about
confidence guarantees, small-sample uncertainty, and plug-in risks before sizing."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
declarative_probabilistic_specification:
  module: "tolerance_intervals_and_decision_sizing"
  domain: "infrastructure_capacity_and_sla_guarantees"
  
  sample_observables:
    sample_size: 15
    sample_mean_ms: 150.0
    sample_std_ms: 20.0
    distribution_family: "Normal(mu, sigma^2)"

  guarantee_requirements:
    coverage_content_P: 0.99      # At least 99% of future requests
    confidence_level_gamma: 0.95  # With 95% statistical confidence
    bound_type: "upper_one_sided"

  computational_methods:
    frequentist_iso_contract:
      standard: "ISO 16269-6"
      method: "non_central_t_exact"
      formula: "T_iso = x_bar + k2(n, P, gamma) * s"
    bayesian_posterior_predictive_contract:
      priors:
        mu: "Normal(mu=150.0, sigma=50.0)"
        sigma: "Uniform(0.0, 100.0)"
      algorithm:
        step_1: "Draw 10,000 posterior pairs (mu^(s), sigma^(s))"
        step_2: "For each pair, compute latent 99th percentile: q_99^(s) = mu^(s) + 2.3263 * sigma^(s)"
        step_3: "Compute 95th percentile of q_99^(s) distribution: T_bayes = quantile(q_99, 0.95)"

  asymmetric_capacity_loss:
    under_provisioning_cost_per_ms: 10000.0  # Massive SLA penalty / dropped user
    over_provisioning_cost_per_ms: 100.0     # Minor idle server hardware cost
    optimal_quantile: "tau_star = 10000 / (10000 + 100) = 0.9901"
```

</details>


## Part 3: The Calibrated Agent Response & Telemetry

When an AI agent receives the Level 1 Micro-Spec from Part 2, it no longer
hallucinates naive plug-in multipliers ($z = 2.326$). Instead, it computes the
exact ISO 16269-6 non-central $t$ tolerance factor and executes Bayesian
posterior predictive sizing, producing the following verified mathematical
telemetry:

### Mathematical Derivation & Exact Tolerance Formulation
1. **The Plug-In Fallacy Defect**:
   Setting
   $$T_{\text{plug}} = \bar{x} + z_P s = 150.0 + 2.3263(20.0) = 196.53\text{ ms}$$
   ignores parameter uncertainty in $\mu$ and $\sigma$. With $n = 15$,
   $\bar{x} \sim \mathcal{N}(\mu, \sigma^2/15)$ and
   $(n-1)s^2/\sigma^2 \sim \chi^2_{14}$. In small samples, this results in
   severe under-provisioning.
2. **ISO 16269-6 Exact Non-Central $t$ Factor**:
   For sample size $n = 15$, coverage $P = 0.99$, and confidence
   $\gamma = 0.95$:
   The one-sided upper tolerance limit factor $k_2$ satisfies:
   $$P_{\bar{X}, S}\left(P_Y(Y \le \bar{X} + k_2 S) \ge P\right) = \gamma$$
   Using the non-central $t$-distribution with $\nu = n - 1 = 14$ degrees of
   freedom and non-centrality parameter
   $\delta = \sqrt{n} z_P = \sqrt{15} \times 2.326348 \approx 9.010$:
   $$k_2 = \frac{t_{\nu, \gamma}(\delta)}{\sqrt{n}} = \frac{t_{14, 0.95}(9.010)}{\sqrt{15}} \approx \mathbf{3.520}$$
   $$T_{\text{ISO}} = \bar{x} + k_2 \cdot s = 150.0 + 3.520 \times 20.0 = \mathbf{220.40\text{ ms}}$$
3. **Bayesian Posterior Predictive Sizing**:
   Under a non-informative Jeffreys prior $p(\mu, \sigma^2) \propto 1/\sigma^2$:
   $$\mu \mid \sigma^2, \mathcal{D} \sim \mathcal{N}\left(\bar{x}, \frac{\sigma^2}{n}\right), \qquad \sigma^2 \mid \mathcal{D} \sim \text{Inv-}\chi^2(n-1, s^2)$$
   The marginal posterior distribution of a future observation $\tilde{y}$
   is a shifted, scaled Student's $t$:
   $$\tilde{y} \mid \mathcal{D} \sim t_{n-1}\left(\bar{x}, s \sqrt{1 + \frac{1}{n}}\right)$$
   Evaluating the posterior predictive distribution at quantile $P = 0.99$
   across posterior draws of $(\mu, \sigma)$ with credibility $\gamma = 0.95$
   yields $T_{\text{Bayes}} \approx 221.12\text{ ms}$.

---

### Experimental Telemetry: Database Microservice Sizing ($n = 15$)
* Sample statistics: $\bar{x} = 150.0\text{ ms}, s = 20.0\text{ ms}$.
* Requirement: Cover $P = 99\%$ of requests with $\gamma = 95\%$ confidence.

```text
================================================================================
           CAPACITY SIZING & TOLERANCE INTERVAL TELEMETRY LOG
================================================================================
Sample Size (n): 15 | Sample Mean (x_bar): 150.00 ms | Sample SD (s): 20.00 ms
Contract: 95% Confidence that >= 99% of future requests complete under T.
--------------------------------------------------------------------------------
METHOD COMPARISON:
  Method 1: Naive Plug-In (x_bar + 2.326*s):           196.53 ms  [FAIL - UNDER]
  Method 2: Frequentist Prediction Interval:        204.81 ms  [NEXT point only]
  Method 3: ISO 16269-6 Tolerance Limit (k=3.520):     220.40 ms  [PASS - EXACT]
  Method 4: Bayesian Posterior Predictive 95% Bound:   221.12 ms  [PASS - BAYES]
--------------------------------------------------------------------------------
MONTE CARLO COVERAGE AUDIT (10,000 SIMULATED FLEETS):
  Method 1 (Naive Plug-In):
    Average fleet coverage achieved:    96.48% (Target: 99.00%)
    Fleets meeting 99% SLA guarantee:   57.62% (Target: 95.00% -- SEVERE BREACH)
    Under-provisioning gap:             -23.87 ms
    
  Method 3 (ISO 16269-6 Non-Central t):
    Average fleet coverage achieved:    99.52%
    Fleets meeting 99% SLA guarantee:   95.14% (Target: 95.00% -- CALIBRATED)
    
  Method 4 (Bayesian Posterior Predictive):
    Average fleet coverage achieved:    99.54%
    Fleets meeting 99% SLA guarantee:   95.31% (Target: 95.00% -- CALIBRATED)
--------------------------------------------------------------------------------
ECONOMIC LOSS AUDIT:
  Capacity Under-Provisioning Cost: $10,000.00 / ms SLA penalty
  Capacity Over-Provisioning Cost:     $100.00 / ms server idle cost
  Optimal Quantile Ratio tau*: 99.01%
  Recommended Timeout Setting: 221.0 ms
================================================================================
```

---

### Visualization: Interval Breadth Comparison

```
              INTERVAL BREADTH AT n = 15 MEASUREMENTS
 Latency
  (ms)
   225 |
   221 |======================================= [Bayesian Bound: 221.1ms]
   220 |--------------------------------------- [ISO 16269-6: 220.4ms]
   215 |
   205 |------------------- [Prediction Interval: 204.8ms]
   196 |========= [Naive Plug-In: 196.5ms (42% SLA FAILURE RATE!)]
   170 |
   150 |● [Sample Mean: 150.0ms]
       +-------------------------------------------------------------
       0        Confidence       Prediction       Tolerance
                Interval         Interval         Interval
```

## Part 4: The 4 Mandatory Verification Gates (How to Audit the Agent)

The AI agent returned a clean, calibrated response: it rejected the plug-in
fallacy, computed the ISO 16269-6 non-central $t$ factor $k_2 = 3.520$,
derived the safe timeout $T = 220.40\text{ ms}$, and validated population
coverage across 10,000 simulated fleets.

However, as an **Epistemic Architect**, you must never accept an AI agent's
capacity calculations at face value. An autonomous agent might quietly confuse
confidence intervals with tolerance limits, use standard normal quantiles,
drop non-centrality shifts, or ignore asymmetric under-provisioning penalties.

Before deploying an automated sizing or timeout recommendation to production
infrastructure, you audit the agent against **four mandatory verification
gates**:

```
              THE 4 MANDATORY AGENTIC VERIFICATION GATES
┌────────────────────────────────────────────────────────────────────┐
│ Gate 1: Prior Predictive Check                                     │
│ Sane physical domain bounds before seeing real data                │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Domain Sanity Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 2: Simulation-Based Calibration (SBC)                         │
│ Algorithmic proof: recovers ground truth across 5,000 simulations  │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Algorithm Soundness Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 3: Mathematical Invariant Conservation                        │
│ Loss asymmetry / $(\beta, \gamma)$ tolerance / flake rate updating │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Loss & Tolerance Invariants Verified]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 4: Posterior Predictive Retrodiction & Loss                   │
│ Sane predictive fit to telemetry & expected loss decision audit    │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │
                                  ▼ [SAFE FOR PRODUCTION DEPLOYMENT]
```

---

### Gate 1: Prior Predictive Check (Physical Latency Domain Sanity)

* **The Production Risk (Why This Gate Exists)**:
  If an agent uses improper flat priors or unbounded diffuse priors (e.g.
  $\mu \sim \mathcal{N}(0, 10^6)$), prior predictive simulations will generate
  negative latencies ($y_{\text{sim}} < 0\text{ ms}$) or unphysical multi-minute
  latency spikes before observing real benchmark data. In capacity planning and
  gateway timeouts, unbounded generative models corrupt log-normal scale
  parameters and create completely unphysical baselines that distort downstream
  SLA bounds.
* **First-Principles Mathematical Mechanism**:
  We evaluate the prior predictive distribution by integrating over the prior
  parameter distributions $p(\mu, \sigma)$:
  $$p(y_{\text{sim}}) = \iint \mathcal{N}(y_{\text{sim}} \mid \mu, \sigma^2) p(\mu, \sigma) \, d\mu \, d\sigma$$
  For domain-calibrated priors (e.g.
  $\mu \sim \mathcal{N}(\mu_0 = 150, \sigma_0^2 = 30^2)$,
  $\sigma \sim \text{Half-Normal}(\sigma_\sigma = 15)$) or log-normal models:
  * We simulate $10{,}000$ synthetic benchmark datasets of size $n = 15$:
    $$\mu^{(s)} \sim p(\mu), \quad \sigma^{(s)} \sim p(\sigma), \quad y_{\text{sim}, i}^{(s)} \sim \mathcal{N}(\mu^{(s)}, (\sigma^{(s)})^2)$$
  * Expected synthetic sample mean:
    $$\mathbb{E}[\bar{y}_{\text{sim}}] = \mu_0 = 150.0\text{ ms}$$
  * Expected synthetic sample standard deviation:
    $$\mathbb{E}[s_{\text{sim}}] \approx 15.0\text{ ms}$$
  * Physical boundary constraint: $P(y_{\text{sim}} < 0) = 0$ strictly.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Non-Negativity Invariant: $\min(y_{\text{sim}}) \ge 0.0\text{ ms}$
    strictly (`[PASS]`).
  * Prior Predictive Mean Invariant:
    $|\bar{y}_{\text{sim}} - 150.0| \le 1.0\text{ ms}$ (`[PASS]`).
  * Dispersion Consistency: $95\%$ of synthetic sample standard deviations
    $s_{\text{sim}}$ fall within $[5.0\text{ ms}, 45.0\text{ ms}]$ (`[PASS]`).
  * Extreme Latency Bound:
    $P(y_{\text{sim}} > 2{,}000\text{ ms}) \le 10^{-4}$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 1 (Prior Predictive Domain Sanity for Sizing):
  Model: n = 15 measurements, Prior mu ~ N(150, 30^2), sigma ~ HalfNormal(15).
  Simulate 10,000 synthetic benchmark datasets:
  1. Draw mu^(s), sigma^(s) from prior.
  2. Simulate 15 latency observations y_sim ~ N(mu^(s), sigma^(s)^2).
  Verify that:
  - 100% of synthetic observations satisfy y_sim >= 0.0 ms.
  - Mean of simulated sample means is 150.0 +/- 1.0 ms.
  - Synthetic standard deviations fall within [5.0, 45.0] ms.
  Report [PASS] with simulated distribution moments or [FAIL] with violations.
  ```

---

### Gate 2: Simulation-Based Calibration (SBC) / Fleet Coverage Proof

* **The Production Risk (Why This Gate Exists)**:
  When an AI agent writes code for tolerance intervals, it routinely defaults
  to the naive plug-in formula ($\bar{x} + z_P s$), or uses incorrect degrees
  of freedom ($n$ instead of $n-1$), or approximates the non-central $t$
  distribution with a central $t$ distribution. In production, this causes
  under-provisioned timeouts, resulting in over $42\%$ of fleets failing to
  achieve their required $99\%$ SLA coverage. Gate 2 mathematically proves
  coverage calibration across thousands of simulated fleets.
* **First-Principles Mathematical Mechanism**:
  SBC runs $M = 5{,}000$ synthetic fleet trials to empirically verify the
  coverage property:
  $$P_{\bar{X}, S}\left( \Phi\left(\frac{T - \mu^*}{\sigma^*}\right) \ge P \right) = \gamma$$
  Algorithm:
  1. Draw ground truth $(\mu^*, \sigma^*) \sim p(\mu, \sigma)$.
  2. Generate synthetic sample of $n = 15$ benchmark measurements:
     $y \sim \mathcal{N}(\mu^*, (\sigma^*)^2)$.
  3. Compute sample statistics $\bar{y} = \frac{1}{n}\sum y_i$ and
     $s = \sqrt{\frac{1}{n-1}\sum (y_i - \bar{y})^2}$.
  4. Compute candidate upper tolerance limit $T = \bar{y} + k \cdot s$.
  5. Calculate exact true population coverage achieved:
     $$C_m = \Phi\left(\frac{T - \mu^*}{\sigma^*}\right)$$
  6. Record coverage success indicator $h_m = \mathbb{I}(C_m \ge 0.99)$.
  7. Compute empirical confidence
     $\hat{\gamma} = \frac{1}{M}\sum_{m=1}^M h_m$.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Empirical Confidence Invariant: Across $M = 5{,}000$ simulated fleets,
    $|\hat{\gamma} - 0.9500| \le 0.0080$ ($[94.20\%, 95.80\%]$) (`[PASS]`).
  * Plug-In Fallacy Rejection: If code sets $k = z_P = 2.326$, immediately raise
    `[FAIL_FATAL_PLUGIN_TRAP]` ($\hat{\gamma}$ will drop to $\approx 57.6\%$).
  * Minimum Multiplier Bound: For $n = 15, P = 0.99, \gamma = 0.95$,
    assert $k_2 \ge 3.40$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 2 (Simulation-Based Calibration on Tolerance Limits):
  Execute 5,000 Monte Carlo fleet sizing trials:
  For m = 1 to 5,000:
    1. Draw mu* ~ N(150, 20^2), sigma* ~ HalfNormal(15).
    2. Generate 15 benchmark measurements y ~ N(mu*, sigma*^2).
    3. Compute x_bar, s, and Upper Tolerance Limit T = x_bar + k2 * s.
    4. Compute true coverage C_m = Phi((T - mu*) / sigma*).
    5. Test if C_m >= 0.99.
  Assert empirical fleet success rate is 95.0% +/- 0.8%.
  Assert k2 multiplier >= 3.40. Report [PASS] or [FAIL].
  ```

---

### Gate 3: Mathematical Invariant Conservation
*($(\beta, \gamma)$ Tolerance & Loss Conservation)*

* **The Production Risk (Why This Gate Exists)**:
  AI agents regularly confuse confidence intervals, prediction intervals, and
tolerance intervals. A confidence interval
($\bar{x} \pm t s / \sqrt{n}$) bounds only the population mean (covering
only ~50% of individual requests). A prediction interval
  prediction interval ($\bar{x} \pm t s \sqrt{1 + 1/n}$) guarantees coverage
  for only the single *next* request. Using either for SLA sizing severely
  under-provisions systems, causing massive production brownouts.
* **First-Principles Mathematical Mechanism**:
  1. **Strict Hierarchy of Interval Widths**:
     For any sample size $n \ge 3$, coverage $P > 0.50$, and confidence
     $\gamma \ge 0.50$:
     $$\text{Width}(\text{CI}) < \text{Width}(\text{PI}) < \text{Width}(\text{TI})$$
     Specifically for $n = 15, P = 0.99, \gamma = 0.95$:
     * Confidence Interval (Mean):
       $$\bar{x} + t_{14, 0.95} \frac{s}{\sqrt{15}} = 150.0 + 1.761 \times \frac{20}{\sqrt{15}} \approx \mathbf{159.09\text{ ms}}$$
     * Prediction Interval (Next single point):
       $$\bar{x} + t_{14, 0.99} s \sqrt{1 + \frac{1}{15}} = 150.0 + 2.624 \times 20 \times 1.0328 \approx \mathbf{204.20\text{ ms}}$$
     * Tolerance Interval (99% fleet coverage with 95% confidence):
       $$\bar{x} + k_2 s = 150.0 + 3.520 \times 20.0 = \mathbf{220.40\text{ ms}}$$
  2. **Asymptotic Monotonicity Invariant**:
     As sample size increases, the tolerance factor $k_2$ decreases
     monotonically toward the normal quantile $z_P$:
     $$\lim_{n \to \infty} k_2(n, P, \gamma) = z_P = \Phi^{-1}(P) = 2.3263, \qquad \frac{\partial k_2}{\partial n} < 0$$
  3. **Economic Asymmetry & Quantile Balancing**:
     With capacity under-provisioning penalty
     $c_{\text{under}} = \$10{,}000/\text{ms}$ and server idle cost
     $c_{\text{over}} = \$100/\text{ms}$:
     $$\tau^* = \frac{c_{\text{under}}}{c_{\text{under}} + c_{\text{over}}} = \frac{10000}{10100} \approx 0.9901 \quad (99.01\%)$$
     The optimal timeout quantile exactly matches the required SLA coverage!
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Interval Strict Ordering: $T_{\text{TI}} > T_{\text{PI}} > T_{\text{CI}}$
    strictly (`[PASS]`).
  * Exact Factor Assertion: $|k_2 - 3.520| \le 0.015$ for
    $n = 15, P = 0.99, \gamma = 0.95$ (`[PASS]`).
  * Asymptotic Monotonicity:
    $k_2(15) > k_2(30) > k_2(100) > 2.3263$ strictly (`[PASS]`).
  * Sizing Deficit Invariant:
    $T_{\text{TI}} - T_{\text{plug}} \ge 20.0\text{ ms}$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 3 (Tolerance Interval Invariant & Hierarchy Check):
  Given n = 15, x_bar = 150 ms, s = 20 ms, P = 0.99, gamma = 0.95:
  1. Compute Upper Confidence Limit (Mean): x_bar + t_crit * s / sqrt(n).
  2. Compute Upper Prediction Limit (Single next point):
     x_bar + t_crit * s * sqrt(1 + 1/n).
  3. Compute ISO Upper Tolerance Limit: x_bar + k2 * s.
  4. Verify strict inequality: UCL_mean < UPL_next < UTL_fleet.
  5. Verify factor bound: k2 >= 3.40 (exact ~ 3.520).
  Assert exact hierarchy and factor boundaries. Report [PASS] or [FAIL].
  ```

---

### Gate 4: Posterior Predictive Retrodiction & Loss Audit

* **The Production Risk (Why This Gate Exists)**:
  Normal tolerance formulas assume Gaussianity. However, production latency
  distributions frequently exhibit heavy tails, bimodal cache behaviors, or
  multi-tenant resource contention. If an agent computes normal tolerance
  limits on heavy-tailed telemetry without running posterior predictive checks,
  tail latencies in production will breach the calculated timeout, resulting in
  cascading downstream gateway timeouts and outage alerts.
* **First-Principles Mathematical Mechanism**:
  1. **Posterior Predictive Replications**:
     Generate $S = 2{,}000$ replicated benchmark datasets
     $\tilde{y}^{(s)} \sim p(\tilde{y} \mid \mathcal{D})$ under the inferred
     posterior:
     $$\mu^{(s)}, \sigma^{(s)} \sim p(\mu, \sigma \mid \mathcal{D}), \quad \tilde{y}^{(s)} \sim \mathcal{N}(\mu^{(s)}, (\sigma^{(s)})^2)$$
  2. **Discrepancy Test Statistics**:
     Define tail skewness statistic $T(y) = \frac{\max(y) - \bar{y}}{s}$.
     Calculate Bayesian posterior predictive p-value:
     $$p_{\text{ppp}} = P(T(\tilde{y}) \ge T(y^{\text{obs}}) \mid \mathcal{D}) = \frac{1}{S} \sum_{s=1}^S \mathbb{I}(T(\tilde{y}^{(s)}) \ge T(y^{\text{obs}}))$$
     If $p_{\text{ppp}} < 0.05$ or $p_{\text{ppp}} > 0.95$, the Gaussian
     assumption is falsified, requiring log-normal or non-parametric
     kernel density estimation.
  3. **Expected Loss Audit**:
     Evaluate total expected operational loss for candidate timeout $T$:
     $$\mathbb{E}[L(T)] = \int \left[ c_{\text{under}} \max(0, y - T) + c_{\text{over}} \max(0, T - y) \right] p(y \mid \mathcal{D}) \, dy$$
     Verify that $T^* \approx 221.0\text{ ms}$ minimizes $\mathbb{E}[L(T)]$.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Retrodictive Discrepancy Range:
    $0.05 \le p_{\text{ppp}} \le 0.95$ (`[PASS]`).
  * Expected Loss Monotonicity:
    $\mathbb{E}[L(T_{\text{ISO}})] < \mathbb{E}[L(T_{\text{plug}})]$
    strictly, with loss savings $\ge \$50{,}000$ across fleet lifetime
    (`[PASS]`).
  * Final Capacity Recommendation: $T \ge 220.0\text{ ms}$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 4 (Posterior Predictive Fit & Loss Sizing Audit):
  Benchmark Telemetry: n = 15, x_bar = 150 ms, s = 20 ms.
  Target: Upper Tolerance Limit T for 99% coverage at 95% confidence.
  1. Generate 2,000 posterior predictive datasets y_rep.
  2. Compute tail discrepancy T(y) = (max(y) - mean(y)) / std(y).
  3. Compute Bayesian p-value p_ppp = P(T(y_rep) >= T(y_obs) | D).
  4. Compute expected loss under c_under = $10,000/ms and c_over = $100/ms.
  5. Assert T_optimal >= 220.0 ms.
  Assert action == SET_TIMEOUT_221MS. Report [PASS] or [FAIL].
  ```

## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Sizing Dimension | Naive AI Assistant Output | Calibrated Probabilistic Specification |
| :--- | :--- | :--- |
| **Statistical Method** | Plug-in fallacy: $\bar{x} + z_P \cdot s$ | ISO 16269-6 Non-central $t$ or Bayesian Posterior Predictive |
| **Multiplier ($n=15$)**| $k = 2.326$ (ignores parameter uncertainty) | $k = 3.520$ (fully accounts for $\mu$ and $\sigma$ noise) |
| **Timeout Limit** | $196.5\text{ ms}$ (under-provisions by $24\text{ms}$) | $220.4\text{ ms}$ (provides certified $99\%$ population coverage) |
| **Empirical SLA Success**| Only $57.6\%$ of fleets meet $99\%$ SLA | Certified $\ge 95\%$ of fleets meet $99\%$ SLA guarantee |
| **Prediction vs Tolerance**| Conflates single next request with population | Explicitly distinguishes prediction vs tolerance bounds |
| **Economic Loss** | Unaware of asymmetric under-provisioning cost | Solves quantile loss: $\tau^* = c_{\text{under}}/(c_{\text{under}}+c_{\text{over}})$ |


## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

Test your skills in auditing AI-generated capacity sizing algorithms.

---

### 🟢 Tier 1: Spot the AI Hallucination
An AI agent calculates memory sizing for a new microservice based on $n = 10$
load test runs ($\bar{x} = 400\text{ MB}, s = 50\text{ MB}$):
> *"To ensure 99% of instances do not crash from OOM, we set container memory to
> $\bar{x} + 2.326 s = 400 + 2.326 \times 50 = 516.3\text{ MB}$."*

**Task**: Identify the mathematical fallacy and estimate the actual risk.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
The agent committed the **Plug-In Fallacy**. With only $n = 10$ samples, the
sample standard deviation $s$ has only 9 degrees of freedom, and the sample mean
has standard error $50 / \sqrt{10} = 15.8\text{ MB}$.

Under ISO 16269-6, the exact tolerance factor for $n = 10, P = 0.99, \gamma =
0.95$ is $k \approx 3.981$.
The true required memory is:
$$\text{Memory}_{\text{ISO}} = 400 + 3.981 \times 50
= \mathbf{599.05\text{ MB}}$$

Allocating $516.3\text{ MB}$ under-provisions memory by over $82\text{ MB}$,
causing Out-Of-Memory container kills on over $10\%$ of production instances.
</details>

---

### 🟡 Tier 2: Write a 3-Sentence Micro-Spec
An engineer needs to size container memory limits for a mission-critical fleet
where OOM kills cause downstream cascades, replacing an ad-hoc standard
deviation multiplier.

**Task**: Write an everyday 3-sentence micro-spec specifying **Data**,
**Prior**, and **Loss/Action**. Then calculate the tolerance bound and
determine the container memory provisioning action.

<details>
<summary><b>Click here to view the solution...</b></summary>

**3-Sentence Micro-Spec**:
1. **Data**: Collect peak memory footprints from $n=15$ stress benchmark runs
   yielding $\bar{x} = 500\text{ MB}$ and $s = 40\text{ MB}$.
2. **Prior**: Place weakly informative log-normal priors on scale parameters
   with safety coverage requirement $P=0.99$ at confidence $\gamma=0.95$.
3. **Loss/Action**: Compute exact upper tolerance factor $k(n=15, P=0.99,
   \gamma=0.95) = 3.520$; provision container memory limit $L = \bar{x} + k
   \cdot s$, logging audit metrics to infrastructure configs.

**Mathematical Calculation & Action Decision**:
* **Tolerance Sizing Calculation**:
  $$L = 500\text{ MB} + 3.520 \times 40\text{ MB}
  = 500 + 140.8 = \mathbf{640.8\text{ MB}}$$
* **Contrast with Naive Normal Cutoff**:
  A naive plug-in approach uses $z_{0.99} = 2.326$:
  $$L_{\text{naive}} = 500 + 2.326 \times 40 = 593.0\text{ MB}$$
  The naive calculation leaves a dangerous $47.8\text{ MB}$ epistemic deficit.
* **Action Decision**: **Action**: Provision container memory limit at
  $641\text{ MB}$ (or standard cloud bucket $650\text{ MB}$) to guarantee $99\%$
  crash-free operation with $95\%$ statistical confidence.
</details>

---

### 🔴 Tier 3: Interrogate the Agent's Verification Gates
An autonomous cloud-cost optimization agent recommends downsizing database pod
memory allocations, claiming a $95\%$ credible interval on average usage proves
the current allocation is wasteful.

**Task**: Formulate a 2-line follow-up audit prompt targeting Gate 2 and Gate 3
failure modes (confusion of confidence/credible interval with tolerance interval
and asymmetric out-of-memory loss neglect). Explain what invariant catches it.

<details>
<summary><b>Click here to view the solution...</b></summary>

**2-Line Follow-Up Audit Prompt**:
> "Clarify whether your bound bounds the mean $\mu$ or covers $99\%$ of
> individual pods ($P=0.99, \gamma=0.95$), and evaluate OOM kills at $50\times$
> idle RAM cost. If you reported a confidence interval for $\mu$ rather than a
> tolerance interval, recompute sizing immediately."

**Invariant & Verification Logic**:
* **Gate 3 Invariant (Tolerance vs. Credible Interval Separation)**: A credible
  interval on $\mu$ shrinks to width zero as $n \to \infty$
  ($\text{SE} = \sigma / \sqrt{n}$), while individual instance variation
  $\sigma$ remains fixed. Sizing container capacity to a confidence interval
  for the mean guarantees that approximately $50\%$ of pods will exceed
  capacity and crash.
* **Gate 2 Invariant (SBC Fleet Coverage Bound)**: Under simulation-based
  calibration, a valid tolerance interval must empirically contain at least
  proportion $P$ of future draws in at least $\gamma$ of simulated fleet trials.
</details>


## Key Takeaways & Course Navigation

### Core Architectural Rules for Capacity & Sizing Systems
1. **Never Use the Plug-In Fallacy ($\bar{x} + z s$)**: The plug-in formula
   ignores parameter uncertainty in $\mu$ and $\sigma$. In small samples, it
   causes massive SLA under-provisioning.
2. **Distinguish the Three Intervals**:
   * *Confidence Interval*: Tracks the mean line (width $\to 0$).
   * *Prediction Interval*: Tracks the single next event.
   * *Tolerance Interval*: Guarantees that $P\%$ of the entire population lives
     inside the bound with $\gamma$ confidence.
3. **Use ISO 16269-6 or Bayesian Posterior Predictive Sizing**: Non-central
   $t$-factors ($k_2$) and Bayesian predictive quantiles correctly expand the
   interval for small samples.
4. **Anchor Sizing in Economic Loss**: Sizing limits are fundamentally
   economic: under-provisioning penalties almost always dwarf idle hardware
   costs.

---

### 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](../README.md)**
* ⬅️ **Previous**: **[Ch 16: Decision Theory & Loss Functions](16_bayesian_decision_theory_and_predictive_checks.ipynb)** *(Part IV: Dynamic Systems, Decision Theory & Predictive Guarantees)*
* ➡️ **Next**: **[Ch 18: Flaky Tests & Pipeline Synthesis](18_case_studies_flaky_tests_and_pipelines.ipynb)** *(Part IV: Dynamic Systems, Decision Theory & Predictive Guarantees)*
* 🔄 **Companion Tracks for Chapter 17**: [📖 Conceptual](../conceptual/17_tolerance_intervals_and_decision_sizing.ipynb) | [🤖 AI Track](17_tolerance_intervals_and_decision_sizing.ipynb) | [🐍 Python](../python/17_tolerance_intervals_and_decision_sizing.ipynb) | [🔵 R](../r/17_tolerance_intervals_and_decision_sizing.ipynb)
